In [2]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.robust.norms as norms
import os
import pandas as pd
import numpy as np
from scipy.optimize import least_squares

Чтение данных

In [3]:
# Путь к директории с CSV-файлами
DATA_DIR = r"C:\proga\ippi\generators\nonlineral\data_outliers"

def parse_filename(filename):
    """
    Извлекает метаинформацию из имени файла.
    
    Структура имени:
      <signal_type>_<params>_<error_category>_<n1>[_<n2>_...].csv
      signal_type:
        l – линия, параметры: a, b (y = a*x + b)
        e – эллипс, параметры: cx, cy, a_axis, b_axis, theta
        p – парабола, параметры: h, k, A 
        h – гипербола, параметры: h, k, a, b
      error_category:
        l – выброс в форме линии
        s – статические выбросы
        r – случайное распределение помех
      После error_category идут от 1 до 5 чисел, каждое задаёт количество точек выброса для группы.
    """

    # Убираем расширение .csv
    name = filename.replace(".csv", "")
    parts = name.split("_")
    
    if not parts:
        raise ValueError("Неверное имя файла: пустая строка.")
    
    # Первый символ — тип сигнала
    signal_type = parts[0]
    meta = {"signal_type": signal_type}
    idx = 1
    
    if signal_type == "l":
        # Ожидается два параметра: a и b
        if len(parts) < idx + 2 + 1:
            raise ValueError(f"Неверное имя файла {filename}: недостаточно параметров для линии.")
        try:
            a = float(parts[idx])
            b = float(parts[idx + 1])
        except ValueError:
            raise ValueError(f"Неверный формат параметров для линии в {filename}.")
        meta["params"] = {"a": a, "b": b}
        idx += 2
    elif signal_type == "e":
        # Эллипс: 5 параметров: cx, cy, a_axis, b_axis, theta
        if len(parts) < idx + 5 + 1:
            raise ValueError(f"Неверное имя файла {filename}: недостаточно параметров для эллипса.")
        try:
            cx, cy, a_axis, b_axis, theta = map(float, parts[idx:idx+5])
        except ValueError:
            raise ValueError(f"Неверный формат параметров для эллипса в {filename}.")
        meta["params"] = {"cx": cx, "cy": cy, "a_axis": a_axis, "b_axis": b_axis, "theta": theta}
        idx += 5
    elif signal_type == "p":
        # Парабола: 3 параметра: h, k, A
        if len(parts) < idx + 3 + 1:
            raise ValueError(f"Неверное имя файла {filename}: недостаточно параметров для параболы.")
        try:
            h, k, A = map(float, parts[idx:idx+3])
        except ValueError:
            raise ValueError(f"Неверный формат параметров для параболы в {filename}.")
        meta["params"] = {"h": h, "k": k, "A": A}
        idx += 3
    elif signal_type == "h":
        # Гипербола: 4 параметра: h, k, a, b
        if len(parts) < idx + 4 + 1:
            raise ValueError(f"Неверное имя файла {filename}: недостаточно параметров для гиперболы.")
        try:
            h_val, k_val, a_val, b_val = map(float, parts[idx:idx+4])
        except ValueError:
            raise ValueError(f"Неверный формат параметров для гиперболы в {filename}.")
        meta["params"] = {"h": h_val, "k": k_val, "a": a_val, "b": b_val}
        idx += 4
    else:
        raise ValueError(f"Неверный тип сигнала в {filename}: {signal_type}")
    
    # Следующий элемент — категория ошибки
    if idx >= len(parts):
        raise ValueError(f"В имени файла {filename} отсутствует категория ошибки.")
    error_category = parts[idx]
    if error_category not in {"l", "s", "r"}:
        raise ValueError(f"Неверная категория ошибки в {filename}: {error_category}")
    meta["error_category"] = error_category
    idx += 1
    
    # Остальные элементы (если есть) — числа, задающие количество точек выбросов для каждой группы
    outlier_counts = []
    while idx < len(parts):
        part = parts[idx].replace(".csv", "")
        try:
            count = int(part)
            outlier_counts.append(count)
        except ValueError:
            raise ValueError(f"Неверное значение количества выбросов в {filename}: {part}")
        idx += 1
    meta["outlier_counts"] = outlier_counts
    
    return meta

def read_data(filepath):
    try:
        data = pd.read_csv(filepath)
        # Если названия колонок не содержат x и y, назначим их
        if not {'x', 'y'}.issubset(data.columns):
            data.columns = ['x', 'y']
        return data['x'].values, data['y'].values
    except Exception as e:
        print(f"Ошибка при чтении файла {filepath}: {e}")
        return None, None

def process_file(filepath, plot = False):
    filename = os.path.basename(filepath)
    try:
        meta = parse_filename(filename)
    except ValueError as e:
        print(e)
        return
    
    x, y = read_data(filepath)
    if x is None or y is None:
        return
    
    if plot:
      print(f"\nОбработка файла: {filename}")
      print("Метаинформация:")
      print(f"  Тип сигнала: {meta['signal_type']}")
      print(f"  Параметры сигнала: {meta['params']}")
      print(f"  Категория ошибки: {meta['error_category']}")
      print(f"  Количество точек выбросов: {meta['outlier_counts']}")
      
      
      
      print(f"Считано точек: {len(x)}")
      # Здесь можно добавить дальнейшую обработку, например, визуализацию или распознавание сигнала.
      plt.figure(figsize=(8, 5))
      plt.scatter(x, y,  color='gray', alpha=0.7, label='Данные с выбросами')
      plt.title(f"Данные: {filename[:-4]}")
      plt.xlabel("x")
      plt.ylabel("y")
      plt.grid(True)
      plt.show()

    return x,y, meta


# process_file("C:\proga\ippi\generators\lin\data_outliers\l_0.57_0.35_s_409_409.csv", True)

Робастные эстиматоры

In [ ]:
def parabola_model_r(params, x, y):
    h, k, A = params
    return (A * (x - h)**2 + k) - y

def ellipse_r(params, x, y):
    cx, cy, a_axis, b_axis, theta = params
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)
    X = (x - cx)*cos_t + (y - cy)*sin_t
    Y = -(x - cx)*sin_t + (y - cy)*cos_t
    return (X / a_axis)**2 + (Y / b_axis)**2 - 1

def hyperbola_r(params, x, y):
    h, k, a, b = params
    return ( (x - h)/a )**2 - ( (y - k)/b )**2 - 1

def robust_regression_curve(x, y, signal_type, method='huber', **kwargs):
    """
    Выполняет робастую оценку параметров кривой второго порядка.
    
    Аргументы:
      x, y       : входные данные (одномерные массивы или списки).
      signal_type: строка, определяющая тип кривой:
                   'p' – парабола, параметры: [h, k, A] (y = A*(x-h)^2 + k);
                   'e' – эллипс, параметры: [cx, cy, a_axis, b_axis, theta],
                         где неявное уравнение: ((x-cx)cosθ+(y-cy)sinθ)^2/a_axis^2 +
                         (-(x-cx)sinθ+(y-cy)cosθ)^2/b_axis^2 = 1;
                   'h' – гипербола, параметры: [h, k, a, b],
                         где неявное уравнение: ((x-h)/a)^2 - ((y-k)/b)^2 = 1.
      method     : функция потерь для least_squares, возможные значения: 
                   'linear' (OLS), 'soft_l1', 'huber', 'cauchy', 'arctan'.
      **kwargs   : дополнительные параметры для least_squares, например max_nfev.
    
      
    """
    x = np.array(x).flatten()
    y = np.array(y).flatten()
    
    loss_option = method.lower() if method.lower() in ['linear','soft_l1','huber','cauchy','arctan'] else 'huber'
    
    if signal_type.lower() == 'p':
        h0 = np.median(x)
        k0 = np.median(y)
        if np.max(x) - np.min(x) != 0:
            A0 = (np.max(y) - np.min(y)) / ((np.max(x) - np.min(x))**2)
        else:
            A0 = 1.0
        initial_guess = [h0, k0, A0]
        res = least_squares(parabola_r, initial_guess, args=(x, y), loss=loss_option, **kwargs)
        return tuple(res.x)
    
    elif signal_type.lower() == 'e':
        # Начальное приближение для эллипса:
        cx0 = np.median(x)
        cy0 = np.median(y)
        a0 = (np.max(x) - np.min(x)) / 4
        b0 = (np.max(y) - np.min(y)) / 4
        theta0 = 0.0  # можно задать 0 или 0.5 радиана
        initial_guess = [cx0, cy0, a0, b0, theta0]
        res = least_squares(ellipse_r, initial_guess, args=(x, y), loss=loss_option, **kwargs)
        return tuple(res.x)
    
    elif signal_type.lower() == 'h':
        # Начальное предположение для гиперболы:
        h0 = np.median(x) - 10
        k0 = np.median(y)
        a0 = (np.max(x) - np.min(x)) / 4
        b0 = (np.max(y) - np.min(y)) / 4
        initial_guess = [h0, k0, a0, b0]
        res = least_squares(hyperbola_r, initial_guess, args=(x, y), loss=loss_option, **kwargs)
        return tuple(res.x)
    
    else:
        raise ValueError("Unsupported signal type for curve fitting")

In [ ]:
class RobustCurveRegressor:
    def __init__(self, model_type='p', loss='huber', **kwargs):
        """
        model_type: 'p' (парабола), 'e' (эллипс), 'h' (гипербола)
        loss: функция потерь для least_squares; возможные значения:
              'linear', 'soft_l1', 'huber', 'cauchy', 'arctan'
        """
        self.model_type = model_type.lower()
        self.loss = loss.lower()
        self.kwargs = kwargs
        self.params_ = None

    def fit(self, x, y, initial_guess=None):
        x = np.array(x).flatten()
        y = np.array(y).flatten()
        if self.model_type == 'p':
            if self.loss in ['linear', 'soft_l1', 'huber', 'cauchy', 'arctan']:
                if initial_guess is None:
                    h0 = np.median(x)
                    k0 = np.median(y)
                    A0 = (np.max(y)-np.min(y))/((np.max(x)-np.min(x))**2 + 1e-8) if len(x) > 1 else 1.0
                    initial_guess = [h0, k0, A0]
                res = least_squares(self._parabola_residuals, initial_guess, args=(x, y),
                                    loss=self.loss, **self.kwargs)
                self.params_ = res.x
            elif self.loss == 'ransac':
                # RANSAC для параболы
                n_samples = len(x)
                best_inliers = -1
                best_params = None
                min_sample = 3  # для параболы нужны минимум 3 точки
                max_trials = self.kwargs.get('max_trials', 100)
                residual_threshold = self.kwargs.get('residual_threshold', 50)
                for _ in range(max_trials):
                    idx = np.random.choice(n_samples, min_sample, replace=False)
                    subset_x = x[idx]
                    subset_y = y[idx]
                    try:
                        reg = RobustCurveRegressor(self.model_type, loss='cauchy')
                        candidate = reg.fit(subset_x, subset_y)
                    except Exception:
                        continue
                    y_pred = candidate._parabola_model(candidate.get_params())
                    inliers = np.sum(np.abs(y - y_pred) < residual_threshold)
                    if inliers > best_inliers:
                        best_inliers = inliers
                        best_params = candidate
                if best_params is None:
                    raise ValueError("RANSAC did not converge")
                y_pred = best_params[2]*(x - best_params[0])**2 + best_params[1]
                inlier_mask = np.abs(y - y_pred) < residual_threshold
                refined = robust_curve_parabola(x[inlier_mask], y[inlier_mask], loss='huber', max_nfev=2000)
                self.params_ = refined
            elif self.loss == 'lms':
                # LMS (Least Median of Squares) для параболы
                n_samples = len(x)
                best_median = np.inf
                best_params = None
                min_sample = 3
                max_trials = self.kwargs.get('max_trials', 1000)
                for _ in range(max_trials):
                    idx = np.random.choice(n_samples, min_sample, replace=False)
                    try:
                        candidate = robust_curve_parabola(x[idx], y[idx], loss='linear', max_nfev=500)
                    except Exception:
                        continue
                    y_pred = candidate[2]*(x - candidate[0])**2 + candidate[1]
                    median_val = np.median((y - y_pred)**2)
                    if median_val < best_median:
                        best_median = median_val
                        best_params = candidate
                if best_params is None:
                    raise ValueError("LMS did not converge")
                self.params_ = best_params
            else:
                raise ValueError(f"Unsupported loss method: {self.loss}")
        elif self.model_type == 'e':
            if self.loss in ['linear', 'soft_l1', 'huber', 'cauchy', 'arctan']:
                if initial_guess is None:
                    cx0 = np.median(x)
                    cy0 = np.median(y)
                    a0 = (np.max(x)-np.min(x))/4
                    b0 = (np.max(y)-np.min(y))/4
                    theta0 = 0.0
                    initial_guess = [cx0, cy0, a0, b0, theta0]
                res = least_squares(self._ellipse_residuals, initial_guess, args=(x, y),
                                    loss=self.loss, **self.kwargs)
                self.params_ = res.x
            elif self.loss == 'ransac':
                """"""
            elif self.loss == 'lms':
                """"""
        elif self.model_type == 'h':
            if self.loss in ['linear', 'soft_l1', 'huber', 'cauchy', 'arctan']:
                if initial_guess is None:
                    h0 = np.median(x) - 10
                    k0 = np.median(y)
                    a0 = (np.max(x)-np.min(x))/4
                    b0 = (np.max(y)-np.min(y))/4
                    initial_guess = [h0, k0, a0, b0]
                res = least_squares(self._hyperbola_residuals, initial_guess, args=(x, y),
                                    loss=self.loss, **self.kwargs)
                self.params_ = res.x
            elif self.loss == 'ransac':
                """"""
            elif self.loss == 'lms':
                """"""
        else:
            raise ValueError(f"Unsupported model type: {self.model_type}")
        return self

    def _parabola_model(self, params, x):
        h, k, A = params
        return A * (x - h)**2 + k

    def _parabola_residuals(self, params, x, y):
        return self._parabola_model(params, x) - y

    def _ellipse_model_implicit(self, params, x, y):
        cx, cy, a_axis, b_axis, theta = params
        cos_t = np.cos(theta)
        sin_t = np.sin(theta)
        Xp = (x - cx) * cos_t + (y - cy) * sin_t
        Yp = -(x - cx) * sin_t + (y - cy) * cos_t
        return (Xp / a_axis)**2 + (Yp / b_axis)**2 - 1

    def _ellipse_residuals(self, params, x, y):
        return self._ellipse_model_implicit(params, x, y)

    def _hyperbola_model_implicit(self, params, x, y):
        h, k, a, b = params
        return ((x - h) / a)**2 - ((y - k) / b)**2 - 1

    def _hyperbola_residuals(self, params, x, y):
        return self._hyperbola_model_implicit(params, x, y)

    def get_params(self):
        return self.params_

def robust_regression_curve(x, y, signal_type='p', method='huber', **kwargs):
    """
    Универсальная функция для робастой регрессии кривых.
    
    Если метод является одним из ['linear','soft_l1','huber','cauchy','arctan'], то используется класс RobustCurveRegressor.
    Если method == 'ransac', выполняется RANSAC.
    Если method == 'LMS', выполняется метод наименьших медиан квадратов.
    
    Возвращает оценённые параметры в виде кортежа.
    """
    x = np.array(x).flatten()
    y = np.array(y).flatten()
    method_lower = method.lower()
    if method_lower in ['linear','soft_l1','huber','cauchy','arctan']:
        regressor = RobustCurveRegressor(model_type=signal_type, method=method_lower, **kwargs)
        regressor.fit(x, y)
        return tuple(regressor.get_params())
    elif method_lower == 'ransac':
        regressor = RobustCurveRegressor(model_type=signal_type, method='ransac', **kwargs)
        regressor.fit(x, y)
        return tuple(regressor.get_params())
    elif method_lower == 'lms':
        regressor = RobustCurveRegressor(model_type=signal_type, method='lms', **kwargs)
        regressor.fit(x, y)
        return tuple(regressor.get_params())
    else:
        raise ValueError("Unsupported method for curve fitting")

Метрика

Обработка всего датасета

График зависимости метрики от сетепени зашумленности данных